[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a1cd/Shears/blob/main/Notebooks/Shears.ipynb)

# Shears: prune and heal a Mixture-of-Experts model

Shears removes experts from a Mixture-of-Experts model and then retrains what is left, with small adapters, so the smaller model behaves like the original again. It is a Swift library on [mlx-swift](https://github.com/ml-explore/mlx-swift); Python here only finds your platform, runs commands and, at the end, loads the result.

This notebook takes Gemma 4 26B-A4B from 128 experts per layer down to 8, heals it to translate English into Japanese, and checks it against the real Gemma's own translations.

**Two modes, chosen in the first code cell**

- `MODEL = "gemma"` (the default): the real model. It downloads about 16 GB, prunes and heals it, exports it, and checks it. An earlier version of this run took about 25 minutes on a Colab T4 (2026-10-22); this version has longer training rows and a longer check, and has not been timed. Choose a GPU runtime first (Runtime, Change runtime type).
- `MODEL = "tiny"`: a random 4-layer model made on the spot. About two minutes on any machine, no GPU. It only shows that every step runs; what it writes is gibberish.

**What you will see.** Before each long step the notebook prints one plain line saying what is happening and about how long it takes, then a status line that updates in place. Set `VERBOSE = True` in the first code cell for every detail. The last section lists how long each phase took.

**The steps, in order**

1. Pick `MODEL`, detect the platform and any GPU, and set up `shears-cli` (built with Xcode on a Mac, a prebuilt download on Linux and Colab).
2. `tiny`: run the pipeline on the random model. `gemma`: download the model, choose the rows to heal on, then prune and heal it block by block and export it.
3. Load the export with Hugging Face transformers and use it: gibberish from the tiny model, or Japanese translations from the pruned Gemma, compared with the real Gemma's.

Run the cells top to bottom. Caveats, data credits and where to go next are at the end.

## Choose what to run

In [ ]:
import os

# "gemma" (default): the real mlx-community/gemma-4-26B-A4B-it-qat-4bit. Downloads about 16 GB, prunes it to 8 experts per layer, heals it for a
#                    few steps and exports it, then checks it as an English to Japanese translator. An earlier version of this run took about 25 minutes on a Colab T4 (2026-10-22); this one is not timed.
# "tiny":            a random 4-layer model made on the spot. Nothing to download; under 2 minutes, on any platform, no GPU needed.
MODEL = "gemma"

# False (default): short output. Each long step prints one plain line (what it does, about how long) and a live status line.
# True: everything. The raw output of every command, CUDA diagnostics, build info, kernel-cache listings, library reports.
VERBOSE = False

# CI sets SHEARS_MODEL, or just runs on GitHub Actions (GITHUB_ACTIONS is set there), and always gets the tiny run.
if os.environ.get("SHEARS_MODEL"):
    MODEL = os.environ["SHEARS_MODEL"]
elif os.environ.get("GITHUB_ACTIONS") == "true":
    MODEL = "tiny"
if os.environ.get("SHEARS_VERBOSE"):
    VERBOSE = os.environ["SHEARS_VERBOSE"] == "1"
assert MODEL in ("gemma", "tiny"), f'MODEL must be "gemma" or "tiny", not {MODEL!r}'
print("MODEL =", MODEL, "| VERBOSE =", VERBOSE)

## 1. Platform and settings

In [ ]:
import os, sys, platform, shutil, subprocess, time, json, pathlib

# --- Settings ---------------------------------------------------------------------------------
SHEARS_GIT_URL = os.environ.get("SHEARS_GIT_URL", "https://github.com/a1cd/Shears.git")   # used only when this notebook is not inside a Shears checkout
SHEARS_REF = os.environ.get("SHEARS_REF", "cuda-binary")            # branch or tag to build in that case
TRANSFORMERS_VERSION = "5.20.2"  # pinned. Gemma 4 first appears in transformers 5.5.0; 5.20.2 is the version tested here
# Linux x86_64: the notebook downloads this prebuilt shears-cli (CPU) and checks its SHA-256, which is read from PREBUILT_URL + ".sha256"
# unless you pin it in PREBUILT_SHA256. A local file works too (a path or a file:// URL). If the download fails the notebook stops
# and says so (docs/linux.md describes building by hand).
PREBUILT_URL = os.environ.get("SHEARS_PREBUILT_URL", "https://github.com/a1cd/Shears/releases/download/linux-x86_64-latest/shears-cli-linux-x86_64.tar.gz")
PREBUILT_SHA256 = os.environ.get("SHEARS_PREBUILT_SHA256", "")
# Linux x86_64 with an NVIDIA GPU (compute capability 7.5 or newer): the notebook first tries this CUDA build. It runs on a Colab T4 (MLX uses the GPU;
# the full pipeline ran end to end there on 2026-10-22). It needs the CUDA libraries a Colab GPU runtime already has (they are not in the tarball).
# If it will not start, the notebook prints why and uses the CPU build above. SHEARS_USE_CUDA=0 skips it.
CUDA_PREBUILT_URL = os.environ.get("SHEARS_CUDA_PREBUILT_URL", "https://github.com/a1cd/Shears/releases/download/linux-x86_64-cuda-experimental/shears-cli-linux-x86_64-cuda.tar.gz")
CUDA_PREBUILT_SHA256 = os.environ.get("SHEARS_CUDA_PREBUILT_SHA256", "")
USE_CUDA_BINARY = os.environ.get("SHEARS_USE_CUDA", "1") != "0"
CCCL_VERSION = "3.1.3"        # the CCCL headers MLX was built with; MLX compiles some kernels at run time against them (CUDA path only)
# MODEL = "gemma":
GEMMA_REPO = "mlx-community/gemma-4-26B-A4B-it-qat-4bit"
GEMMA_STEPS_PER_BLOCK = 10     # healing steps for each of the 5 blocks of layers (export needs adapters for every layer, so every block is healed)
TRAIN_ROWS = 32                # English sentences (with a human Japanese translation) to heal on
CHECK_ROWS = 50                # held-out sentences the pruned model is checked on at the end (the file has 200; more rows = longer check)
SHOW_ROWS = 5                  # of those, how many are also translated by the pruned model and shown next to Gemma's translation
# The translator task. The data files are in the repo's Notebooks/data/ (the notebook reads them from its copy of the sources).
SYSTEM_PROMPT_FILE = "Notebooks/data/translate-system-prompt-ja.txt"   # the system prompt, every training row and every check uses it verbatim
GEMMA_DATA = "Notebooks/data/gemma-en-ja-200.jsonl"                    # 200 English sentences with Gemma 4 26B-A4B's own Japanese translations (see Notebooks/data/README.md)
ALLOW_GEMMA_ON_CPU = False     # if MLX ends up on the CPU (no usable GPU) the notebook stops before the 16 GB download; True goes on anyway

# --- Platform ---------------------------------------------------------------------------------
IS_MAC = sys.platform == "darwin"
IS_LINUX = sys.platform.startswith("linux")
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ARCH = platform.machine()
if IS_MAC and ARCH != "arm64":
    raise SystemExit("Shears needs Apple Silicon (MLX does not run on Intel Macs). Use Colab or Linux instead.")
if not (IS_MAC or IS_LINUX):
    raise SystemExit(f"Unsupported platform: {sys.platform}. Use an Apple Silicon Mac or x86_64 Linux.")
print(f"platform: {'Apple Silicon Mac' if IS_MAC else 'Linux'} ({ARCH}){', Colab' if IN_COLAB else ''}, Python {platform.python_version()}")
if IS_LINUX and VERBOSE:
    print(open("/etc/os-release").read().split("PRETTY_NAME=")[1].splitlines()[0], "| CPUs:", os.cpu_count())

def detect_nvidia():
    """(info, why_not): info is {"name", "cc", "mem_mib", "driver"} for GPU 0 from nvidia-smi, else None and the reason."""
    exe = shutil.which("nvidia-smi")
    if not exe:
        return None, "nvidia-smi not found: no NVIDIA GPU on this runtime"
    r = subprocess.run([exe, "--query-gpu=name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        return None, f"nvidia-smi failed (exit {r.returncode}): {(r.stderr or r.stdout).strip()[:200]}"
    f = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
    try:
        return {"name": f[0], "cc": float(f[1]), "mem_mib": int(float(f[2])), "driver": f[3]}, ""
    except (IndexError, ValueError):
        return None, f"could not read nvidia-smi output: {r.stdout.strip()[:200]}"

GPU, GPU_WHY_NOT = detect_nvidia() if IS_LINUX else (None, "not Linux")
if IS_LINUX:
    print((f"NVIDIA GPU: {GPU['name']}, {GPU['mem_mib'] / 1024:.0f} GB" + (f" (compute capability {GPU['cc']}, driver {GPU['driver']})" if VERBOSE else "")) if GPU else f"NVIDIA GPU: none ({GPU_WHY_NOT})")

# --- Where things live ------------------------------------------------------------------------
# SwiftPM names a path dependency after its directory, so the sources must sit in a directory called "Shears".
WORK = pathlib.Path(os.environ.get("SHEARS_WORK") or ("/content/shears-work" if IN_COLAB else pathlib.Path.home() / "shears-notebook-work"))
SRC = WORK / "Shears"
WORK.mkdir(parents=True, exist_ok=True)

# Inside a checkout (Notebooks/ next to Package.swift)? Use that code. Otherwise clone it.
here = pathlib.Path.cwd()
LOCAL_REPO = next((p for p in (here.parent, here) if (p / "Package.swift").exists() and (p / "Sources" / "Shears").is_dir()), None)

# --- Helpers ----------------------------------------------------------------------------------
PHASES = []   # (name, seconds)

def vprint(*args, **kw):
    """print, but only when VERBOSE = True"""
    if VERBOSE:
        print(*args, **kw)

def step(text, t4=None):
    """One plain line before a long step: what happens and about how long. The times are from the Colab T4 run of 2026-10-22."""
    print(f"==> {text}" + (f" (about {t4} on the Colab T4 this was timed on; other hardware differs)" if t4 and not IS_MAC else ""))

def sh(script, name, cwd=None, env=None, quiet=False, on_line=None, log=None):
    """Run a bash script (stops at the first error), stream its output, and record how long it took.
    quiet=True prints nothing, or the last 60 lines if it fails (for noisy builds and installs).
    on_line=f: f gets every output line (for a live status line) and the lines are not printed, unless VERBOSE. log=path keeps every line in a file;
    when the command fails the last 40 lines are printed and the log's path."""
    e = dict(os.environ, **(env or {}))
    t0 = time.time()
    p = subprocess.Popen(["bash", "-c", "set -eo pipefail\n" + script], cwd=cwd, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    parsed = on_line is not None and not VERBOSE
    logf = open(log, "w") if log else None
    tail = []
    for line in p.stdout:
        if logf:
            logf.write(line)
        if parsed:
            tail = (tail + [line])[-40:]
            try:
                on_line(line.rstrip("\n"))
            except Exception:
                pass          # a status line must never stop the run
        elif quiet:
            tail = (tail + [line])[-60:]
        else:
            print(line, end="")
    rc = p.wait()
    if logf:
        logf.close()
    dt = time.time() - t0
    PHASES.append((name, dt))
    if rc != 0 and (parsed or quiet):
        print("".join(tail), end="")
    if rc != 0 and log:
        print(f"(the full output is in {log})")
    if rc != 0 or VERBOSE or not (parsed or quiet):
        print(f"--- {name}: {dt:.1f} s")
    if rc != 0:
        raise RuntimeError(f"{name} failed with exit status {rc}")

def add_to_path(*dirs):
    os.environ["PATH"] = os.pathsep.join([str(d) for d in dirs] + [os.environ["PATH"]])

def phase_table():
    for n, s in PHASES:
        print(f"{n:42} {s:8.1f} s")
    print(f"{'total':42} {sum(s for _, s in PHASES):8.1f} s")

vprint("work directory:", WORK)
vprint("Shears code from:", LOCAL_REPO if LOCAL_REPO else f"{SHEARS_GIT_URL} @ {SHEARS_REF}")

### 1b. Helpers, and (optional) CUDA diagnostics

Helper code for the downloads and the status lines lives in the next cell. With `VERBOSE = True` it also prints a diagnostic report for a machine with an NVIDIA GPU: nvidia-smi, the CUDA libraries and their versions, `ldd` of the binary, and a traced run of `shears-cli device`. That report is also printed on its own when the CUDA build fails to start, whatever `VERBOSE` says. `CUDA_DIAGNOSTICS_ONLY = True` in the cell stops right after it.

In [ ]:
# 1b. CUDA diagnostics (and the helpers section 2b uses). On a machine with an NVIDIA GPU this takes about a minute and prints everything needed to
# debug the CUDA build: versions, libraries, the library check, and a traced run of `shears-cli device`. Nothing here builds or downloads a model.
CUDA_DIAGNOSTICS = True        # with VERBOSE = True (or CUDA_DIAGNOSTICS_ONLY), run the checks below now (no effect without an NVIDIA GPU)
CUDA_DIAGNOSTICS_ONLY = False  # True: run them and stop right after (a ~3 minute session: set MODEL = "tiny", this True, run the cells down to here)

import hashlib, tarfile, urllib.request, zipfile, glob, re, tempfile, signal, threading

class Progress:
    """One line that updates in place: bytes so far, total, speed, time left. Shows that a download is moving."""
    def __init__(self, total, label, start=0):
        self.total, self.label, self.t0, self.last, self.start = total, label, time.time(), 0.0, start
        try:
            from IPython.display import display, Pretty
            self.handle = display(Pretty(self._text(0)), display_id=True)
        except Exception:
            self.handle = None
    def _text(self, n):
        dt = max(time.time() - self.t0, 1e-6)
        rate = max(n - self.start, 0) / dt
        parts = [f"{self.label}: {n / 1e9:.2f} GB"]
        if self.total:
            pct = 100 * n / self.total
            filled = int(pct / 4)
            parts = [f"{self.label}: [{'#' * filled}{'.' * (25 - filled)}] {pct:5.1f}%  {n / 1e9:.2f} / {self.total / 1e9:.2f} GB"]
            if rate > 0:
                left = (self.total - n) / rate
                parts.append(f"ETA {int(left // 60)}m{int(left % 60):02d}s")
        parts.append(f"{rate / 1e6:.0f} MB/s")
        return "  ".join(parts)
    def update(self, n, force=False):
        if force or time.time() - self.last > 0.5:
            self.last = time.time()
            if self.handle is not None:
                from IPython.display import Pretty
                self.handle.update(Pretty(self._text(n)))
    def done(self, n):
        self.update(n, force=True)
        if self.handle is None:
            print(self._text(n))

class Status:
    """One line that updates in place, like the download bars: a short text and, with ticker=True, the time since it started (so a long
    quiet step still shows it is alive). Without IPython (a plain terminal) only the final text is printed."""
    def __init__(self, text, ticker=True):
        self.text, self.t0, self.ticker, self.stopped = text, time.time(), ticker, threading.Event()
        try:
            from IPython import get_ipython
            from IPython.display import display, Pretty
            assert get_ipython() is not None          # a plain terminal would print every update as a new line
            self.handle = display(Pretty(self._render()), display_id=True)
        except Exception:
            self.handle = None
        if self.handle is not None and ticker:
            threading.Thread(target=self._tick, daemon=True).start()
    def _render(self):
        t = int(time.time() - self.t0)
        return self.text + (f"   [{t // 60}:{t % 60:02d} elapsed]" if self.ticker else "")
    def _tick(self):
        while not self.stopped.wait(1.0):
            self._show()
    def _show(self):
        try:
            from IPython.display import Pretty
            self.handle.update(Pretty(self._render()))
        except Exception:
            pass
    def set(self, text):
        self.text = text
        if self.handle is not None:
            self._show()
    def done(self, text=None):
        """Stop the clock and leave the last text (or `text`) on screen."""
        if text is not None:
            self.text = text
        self.stopped.set()
        self.ticker = False
        if self.handle is not None:
            self._show()
        else:
            print(self.text)

def show_table(headers, rows, caption=None, max_height=None):
    """A table of text. In a notebook it is an HTML table (long texts wrap, so the columns sit side by side); elsewhere plain text blocks."""
    try:
        import html
        from IPython import get_ipython
        from IPython.display import display, HTML
        assert get_ipython() is not None
        h = "<table style='text-align:left;vertical-align:top;border-collapse:collapse'>"
        h += "<tr>" + "".join(f"<th style='padding:4px 10px;border-bottom:1px solid #888'>{html.escape(x)}</th>" for x in headers) + "</tr>"
        for r in rows:
            h += "<tr>" + "".join(f"<td style='padding:4px 10px;vertical-align:top;border-bottom:1px solid #ddd'>{html.escape(str(x))}</td>" for x in r) + "</tr>"
        h += "</table>"
        if max_height:
            h = f"<div style='max-height:{max_height}px;overflow-y:auto'>{h}</div>"
        display(HTML((f"<b>{html.escape(caption)}</b><br>" if caption else "") + h))
    except Exception:
        if caption:
            print(caption)
        for r in rows:
            print("\n".join(f"  {a}: {b}" for a, b in zip(headers, r)))
            print()

def fetch(url, dest):
    if "://" not in url:
        url = pathlib.Path(url).resolve().as_uri()
    req = urllib.request.Request(url, headers={"User-Agent": "shears-notebook"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        bar = Progress(total, pathlib.Path(dest).name)
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            bar.update(f.tell())
        bar.done(f.tell())

def fetch_and_unpack(url, pinned_sha, dest):
    """Download url (and, unless pinned_sha is given, url + '.sha256'), check the SHA-256, unpack into dest, return bin/shears-cli.
    A copy already unpacked from the same url is reused."""
    marker = dest / ".from-url"
    if marker.exists() and marker.read_text() == url:
        found = next((dest / "pkg").glob("*/bin/shears-cli"), None)
        if found:
            vprint("already downloaded:", found)
            return found
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    vprint("downloading", url)
    fetch(url, dest / "shears-cli.tar.gz")
    want = pinned_sha.strip().lower()
    if not want:
        fetch(url + ".sha256", dest / "shears-cli.tar.gz.sha256")
        want = (dest / "shears-cli.tar.gz.sha256").read_text().split()[0].lower()
    h = hashlib.sha256()
    with open(dest / "shears-cli.tar.gz", "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    if h.hexdigest() != want:
        raise ValueError(f"SHA-256 mismatch: expected {want}, got {h.hexdigest()}")
    vprint("SHA-256 ok:", want, f"({(dest / 'shears-cli.tar.gz').stat().st_size / 1e6:.1f} MB)")
    with tarfile.open(dest / "shears-cli.tar.gz") as tf:
        if hasattr(tarfile, "data_filter"):
            tf.extractall(dest / "pkg", filter="data")
        else:
            tf.extractall(dest / "pkg")
    marker.write_text(url)
    return next((dest / "pkg").glob("*/bin/shears-cli"))

def install_gxx():
    # g++ is needed at run time: MLX compiles fused CPU kernels with it.
    sh(r"""
SUDO=""; [ "$(id -u)" = 0 ] || SUDO=sudo
if ! command -v g++ >/dev/null; then
  export DEBIAN_FRONTEND=noninteractive
  $SUDO apt-get update -qq && $SUDO apt-get install -y -qq g++
fi
g++ --version | head -1
""", "g++ for MLX's CPU kernels", quiet=not VERBOSE)

def nvidia_library_dirs():
    """Directories where the NVIDIA libraries may live on this machine: the toolkit and the driver first, then pip wheels."""
    system = []
    for pat in ["/usr/local/cuda/lib64", "/usr/local/cuda-*/lib64", "/usr/local/cuda-*/targets/x86_64-linux/lib",
                "/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib64-nvidia", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"]:
        system += glob.glob(pat)
    import site, sysconfig
    wheels = []
    for sp in {*site.getsitepackages(), sysconfig.get_paths()["purelib"]}:
        wheels += sorted({os.path.dirname(p) for p in glob.glob(os.path.join(sp, "nvidia", "**", "lib*.so*"), recursive=True)})
    return list(dict.fromkeys(system)), list(dict.fromkeys(wheels))

def missing_libraries(cli, ld_path):
    r = subprocess.run(["ldd", str(cli)], capture_output=True, text=True, env=dict(os.environ, LD_LIBRARY_PATH=ld_path))
    return [l.split("=>")[0].strip() for l in r.stdout.splitlines() if "not found" in l]

def resolve_nvidia_libraries(cli, allow_pip=True):
    """Find the NVIDIA libraries `cli` needs (they are not in its tarball). Returns (LD_LIBRARY_PATH, still-missing library names)."""
    system, wheels = nvidia_library_dirs()
    ld = ":".join(system + [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d])
    def missing(): return [m for m in missing_libraries(cli, ld) if not m.startswith("libcuda.so")]
    def add_from(wheels_, names):
        nonlocal ld
        for m in names:
            hits = sorted([d for d in wheels_ if glob.glob(os.path.join(d, m))], key=lambda d: "cu13" not in d)   # a CUDA 13 wheel before a CUDA 12 one
            if hits:
                ld = hits[0] + ":" + ld
    names = missing()
    if names:
        print("libraries not on the system path:", names, "- looking in pip's nvidia wheels")
        add_from(wheels, names)
        names = missing()
    if names and allow_pip:
        print("still missing", names, "- trying pip install of the CUDA 13 runtime wheels")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "nvidia-cuda-runtime",
                        "nvidia-cublas", "nvidia-cuda-nvrtc", "nvidia-cufft", "nvidia-cudnn-cu13"], capture_output=True, text=True)
        add_from(nvidia_library_dirs()[1], names)
        names = missing()
    return ld, names

def exit_description(rc):
    if rc is not None and rc < 0:
        try:
            return f"killed by signal {-rc} ({signal.Signals(-rc).name})"
        except ValueError:
            return f"killed by signal {-rc}"
    return f"exit status {rc}"

LIBRARY_VERSION_PROBE = r"""
import ctypes
def load(name):
    try:
        return ctypes.CDLL(name)
    except OSError as e:
        print(f"  {name}: NOT LOADABLE: {e}"); return None
cu = load("libcuda.so.1")
if cu:
    print("  libcuda.so.1: cuInit ->", cu.cuInit(0))
    v = ctypes.c_int(); cu.cuDriverGetVersion(ctypes.byref(v)); print("  driver supports CUDA", v.value)
    n = ctypes.c_int(); cu.cuDeviceGetCount(ctypes.byref(n)); print("  cuDeviceGetCount ->", n.value)
rt = load("libcudart.so.13")
if rt:
    v = ctypes.c_int(); rt.cudaRuntimeGetVersion(ctypes.byref(v)); print("  libcudart.so.13: runtime", v.value)
    rc = rt.cudaFree(0); rt.cudaGetErrorString.restype = ctypes.c_char_p; print("  cudaFree(0) (creates the context) ->", rc, rt.cudaGetErrorString(rc))
nv = load("libnvrtc.so.13")
if nv:
    a, b = ctypes.c_int(), ctypes.c_int(); nv.nvrtcVersion(ctypes.byref(a), ctypes.byref(b)); print(f"  libnvrtc.so.13: {a.value}.{b.value}")
lt = load("libcublasLt.so.13")
if lt:
    lt.cublasLtGetVersion.restype = ctypes.c_size_t; print("  libcublasLt.so.13: version", lt.cublasLtGetVersion())
load("libcublas.so.13")
dn = load("libcudnn.so.9")
if dn:
    dn.cudnnGetVersion.restype = ctypes.c_size_t; print("  libcudnn.so.9: version", dn.cudnnGetVersion())
ff = load("libcufft.so.12")
if ff:
    v = ctypes.c_int(); ff.cufftGetVersion(ctypes.byref(v)); print("  libcufft.so.12: version", v.value)
"""

def cuda_diagnostics(cli=None, ld=None, probe=None):
    """Print what is needed to debug the CUDA build. `probe` is an already finished CompletedProcess of `shears-cli device`, if any."""
    def run(cmd, **kw):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=600, **kw)
            return (r.stdout + r.stderr).rstrip()
        except Exception as e:
            return f"({type(e).__name__}: {e})"
    print("=" * 24, "CUDA diagnostics", "=" * 24)
    print("--- nvidia-smi"); print("\n".join(run(["nvidia-smi"]).splitlines()[:14]))
    nvcc = shutil.which("nvcc") or next(iter(glob.glob("/usr/local/cuda*/bin/nvcc")), None)
    print("--- nvcc:", (run([nvcc, "--version"]).splitlines() or ["?"])[-1] if nvcc else "none (not needed to run the binary)")
    print("--- LD_LIBRARY_PATH (this kernel):", os.environ.get("LD_LIBRARY_PATH", "(unset)"))
    print("--- the CUDA libraries on this machine (ls /usr/local/cuda*/lib64 | grep -E 'cudart|cublas|cudnn|nvrtc|cufft', and the others)")
    pat = re.compile(r"cudart|cublas|cudnn|nvrtc|cufft|libcuda\.")
    places = glob.glob("/usr/local/cuda*/lib64") + glob.glob("/usr/local/cuda*/targets/*/lib") + ["/usr/lib64-nvidia", "/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu"] + nvidia_library_dirs()[1]
    for d in dict.fromkeys(places):
        names = sorted(n for n in (os.listdir(d) if os.path.isdir(d) else []) if pat.search(n) and ".so" in n)
        if names:
            print(f"  {d}: {' '.join(names[:40])}" + (" ..." if len(names) > 40 else ""))
    if cli is None:
        print("(no shears-cli yet)"); return
    root = cli.parent.parent
    if ld is None:
        ld, _ = resolve_nvidia_libraries(cli, allow_pip=False)
    print("--- BUILD_INFO.txt"); print("  " + (root / "BUILD_INFO.txt").read_text().strip().replace("\n", "\n  "))
    print("--- LD_LIBRARY_PATH used for the probe:", ld)
    print("--- ldd shears-cli   (lines with 'not found' are marked)")
    for l in run(["ldd", str(cli)], env=dict(os.environ, LD_LIBRARY_PATH=ld)).splitlines():
        print(("  !!! " if "not found" in l else "      ") + l.strip())
    print("--- the NVIDIA libraries asked for, loaded by python (versions):")
    print(run([sys.executable, "-c", LIBRARY_VERSION_PROBE], env=dict(os.environ, LD_LIBRARY_PATH=ld)))
    sys_cxx = [q for q in ("/usr/lib/x86_64-linux-gnu/libstdc++.so.6", "/usr/lib/x86_64-linux-gnu/libgcc_s.so.1") if os.path.exists(q)]
    variants = [("shears-cli device", {}),
                ("shears-cli device, leaving the ordinary way (SHEARS_NORMAL_EXIT=1: runs C++ static destructors at exit)", {"SHEARS_NORMAL_EXIT": "1"}),
                ("shears-cli device with CUDA_LAUNCH_BLOCKING=1", {"CUDA_LAUNCH_BLOCKING": "1"}),
                ("shears-cli device with the system libstdc++ preloaded (rules out two copies of it)", {"LD_PRELOAD": ":".join(sys_cxx)})]
    for label, extra in variants:
        if probe is not None and not extra:
            r = probe
        else:
            r = subprocess.run([str(cli), "device"], capture_output=True, text=True, timeout=900, env=dict(os.environ, LD_LIBRARY_PATH=ld, **extra))
        print(f"--- {label}: {exit_description(r.returncode)}")
        print("  stdout:", r.stdout.rstrip() or "(empty)")
        print("  stderr (all of it):"); print("\n".join("    " + l for l in (r.stderr.rstrip() or "(empty)").splitlines()))
        if r.returncode == 0:
            break
    print("=" * 66)

if IS_LINUX and GPU and USE_CUDA_BINARY and CUDA_DIAGNOSTICS and (VERBOSE or CUDA_DIAGNOSTICS_ONLY):
    _cli = fetch_and_unpack(CUDA_PREBUILT_URL, CUDA_PREBUILT_SHA256, WORK / "prebuilt-cuda")
    cuda_diagnostics(_cli)
    if CUDA_DIAGNOSTICS_ONLY:
        raise SystemExit("Stopping after the CUDA diagnostics (CUDA_DIAGNOSTICS_ONLY = True).")
elif CUDA_DIAGNOSTICS and VERBOSE:
    print("CUDA diagnostics: skipped (" + (GPU_WHY_NOT if not GPU else "SHEARS_USE_CUDA=0") + ")")
def jit_cache_report(pack=False):
    """What MLX compiled at run time on this GPU (NVRTC) and kept: one .ptx file (cubin) per module. One line; VERBOSE lists the files and the gaps
    between compiles (their modification times). With pack=True also writes a tarball of the cache."""
    d = os.environ.get("MLX_PTX_CACHE_DIR")
    if not d or not os.path.isdir(d):
        vprint("MLX run-time kernel cache: none (the CPU build, or nothing compiled yet)")
        return
    files = sorted((f for f in pathlib.Path(d).rglob("*.ptx")), key=lambda f: f.stat().st_mtime)
    line = f"GPU kernels MLX compiled during this run: {len(files)}, {sum(f.stat().st_size for f in files) / 1e6:.1f} MB (cache: {d})"
    if VERBOSE:
        print(line)
    if not files:
        if not VERBOSE:
            print(line)
        return
    gaps = [b.stat().st_mtime - a.stat().st_mtime for a, b in zip(files, files[1:])]
    if gaps and VERBOSE:
        print(f"  gaps between consecutive compiles: longest {max(gaps):.0f} s, median {sorted(gaps)[len(gaps) // 2]:.0f} s, total {sum(gaps):.0f} s (includes the work between them)")
    if VERBOSE:
        for f in files[:40]:
            print(f"  {f.stat().st_size / 1e3:9.1f} kB  {f.name}")
        if len(files) > 40:
            print(f"  ... and {len(files) - 40} more")
    if pack:
        out = WORK / f"jit-cache-{os.path.basename(d)}.tar.gz"
        with tarfile.open(out, "w:gz") as tf:
            for f in pathlib.Path(d).iterdir():
                tf.add(f, arcname=f.name)
        if not VERBOSE:
            print(line.split(" (cache:")[0] + f"; saved as {out.name} ({out.stat().st_size / 1e6:.1f} MB)")
        print(f"A later session on the same kind of GPU can set SHEARS_JIT_CACHE_URL to a copy of {out.name} (or SHEARS_JIT_CACHE to a folder with the unpacked files) and skip the first-use compiles. "
              "It holds code for ONE compute capability and ONE build of shears-cli." if VERBOSE else f"Saved to {out}. Next time, SHEARS_JIT_CACHE_URL set to a copy of it skips the first-use compiles (same GPU type and same shears-cli build only).")
    elif not VERBOSE:
        print(line.split(" (cache:")[0])

## 2. Setup: get `shears-cli`

First the Shears sources are copied into the working folder (the pipeline reads its example config and the data files from there). Then `shears-cli` itself: on a Mac it is built with `xcodebuild` (2a); on x86_64 Linux and Colab the prebuilt binary is downloaded and its checksum checked (2b). The cell for the other platform just says it is skipping.

In [ ]:
# Put the sources in WORK/Shears.
if LOCAL_REPO:
    if SRC.exists():
        shutil.rmtree(SRC)
    shutil.copytree(LOCAL_REPO, SRC, ignore=shutil.ignore_patterns(".git", ".build", "Packages", ".swiftpm", "DerivedData", "*.xcodeproj", "__pycache__", ".ipynb_checkpoints"))
    vprint("copied", LOCAL_REPO, "->", SRC)
elif not (SRC / "Package.swift").exists():
    sh(f'git clone --depth 1 --branch "{SHEARS_REF}" "{SHEARS_GIT_URL}" "{SRC}"', "git clone Shears", quiet=not VERBOSE)
sh("git log -1 --format='sources at %h %s' 2>/dev/null || true", "sources", cwd=SRC, quiet=not VERBOSE)

### 2a. Apple Silicon Mac

Needs Xcode with the Metal Toolchain (`xcodebuild -downloadComponent MetalToolchain` if the build complains about `metal`). Plain `swift build` cannot compile MLX's Metal shaders, so this uses `xcodebuild`.

In [ ]:
if IS_MAC:
    sh("xcrun -f metal >/dev/null 2>&1 || { echo 'No Metal Toolchain. Run: xcodebuild -downloadComponent MetalToolchain'; exit 1; }", "check Metal toolchain", quiet=not VERBOSE)
    sh(f'cd "{SRC}/CLI" && xcodebuild build -scheme shears-cli -destination "platform=macOS" -configuration Release '
       f'-derivedDataPath "{WORK}/derived" -quiet 2>&1', "xcodebuild shears-cli", quiet=True)
    SHEARS_CLI = WORK / "derived/Build/Products/Release/shears-cli"
else:
    print("not a Mac: skipping (the Linux cells below do the setup)")

### 2b. Linux x86_64 and Colab: the prebuilt `shears-cli`

This cell downloads a prebuilt `shears-cli`, checks its SHA-256, runs it once as a test, installs `g++` if needed (MLX compiles some CPU kernels with it) and puts it on `PATH`. With an NVIDIA GPU of compute capability 7.5 or newer (a Colab T4 is one) it uses the **CUDA build**, so MLX runs on the GPU. Without one, or if the CUDA build will not start, it prints why and uses the **CPU build**. If neither can be set up it stops and says why; [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md) describes building by hand.

In [ ]:
PREBUILT_OK = False     # a prebuilt shears-cli (CUDA or CPU) was set up
CUDA_OK = False         # ... and it is the CUDA one, and MLX runs a real operation on the GPU with it
CUDA_REASON = None      # why the CUDA build was not used, when it was skipped or failed
if IS_LINUX and ARCH != "x86_64":
    raise RuntimeError(f"There is no prebuilt shears-cli for Linux {ARCH}. Build it by hand: https://github.com/a1cd/Shears/blob/main/docs/linux.md")
if IS_LINUX:
    # ---------------------------------------------------------------------------------------------
    # The CUDA build, if there is a GPU it was built for. Any failure here is a printed reason, then the CPU build.
    # (The helpers are defined in section 1b.)
    # ---------------------------------------------------------------------------------------------
    def try_cuda_build():
        """Set up the CUDA build and prove it runs an operation on the GPU. Returns the path of shears-cli, or raises with the reason."""
        if not USE_CUDA_BINARY:
            raise RuntimeError("SHEARS_USE_CUDA=0")
        if not GPU:
            raise RuntimeError(GPU_WHY_NOT)
        if GPU["cc"] < 7.5:
            raise RuntimeError(f"{GPU['name']} has compute capability {GPU['cc']}; the CUDA build needs 7.5 or newer (MLX's own minimum)")
        t1 = time.time()
        cli = fetch_and_unpack(CUDA_PREBUILT_URL, CUDA_PREBUILT_SHA256, WORK / "prebuilt-cuda")
        vprint("CUDA build:", (cli.parent.parent / "BUILD_INFO.txt").read_text().strip().replace("\n", "\n  "))
        # 1. The NVIDIA libraries are not in the tarball. Find them.
        ld, missing = resolve_nvidia_libraries(cli)
        if missing:
            cuda_diagnostics(cli, ld)
            raise RuntimeError(f"these CUDA libraries are not on this machine: {missing}")
        if "libcuda.so.1" in " ".join(missing_libraries(cli, ld)):
            cuda_diagnostics(cli, ld)
            raise RuntimeError("libcuda.so.1 (the NVIDIA driver library) is not found although nvidia-smi works")
        # 2. MLX compiles some kernels at run time (NVRTC, first use of a GEMM or a fused op) against CUTLASS/CuTe and CCCL headers it looks for in
        #    <bundle>/include and <bundle>/include/cccl. The tarball carries them. (A tarball without them failed on a T4 with: cannot open source file "cute/tensor.hpp".)
        inc = cli.parent.parent / "include"
        if not (inc / "cute" / "tensor.hpp").exists() or not (inc / "cutlass" / "cutlass.h").exists():
            raise RuntimeError(f"this CUDA tarball has no CUTLASS/CuTe headers in {inc} (MLX's run-time kernels need them): use a newer one")
        if not (inc / "cccl" / "cuda").is_dir():
            try:
                z = WORK / "prebuilt-cuda" / "cccl.zip"
                fetch(f"https://github.com/NVIDIA/cccl/releases/download/v{CCCL_VERSION}/cccl-v{CCCL_VERSION}.zip", z)
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(WORK / "prebuilt-cuda" / "cccl")
                root = next(p for p in [*(WORK / "prebuilt-cuda" / "cccl").glob("*/include"), WORK / "prebuilt-cuda" / "cccl" / "include"] if p.is_dir())
                for sub in ("cuda", "nv"):
                    shutil.copytree(root / sub, inc / "cccl" / sub, dirs_exist_ok=True)
                print(f"CCCL {CCCL_VERSION} headers fetched into", inc / "cccl")
            except Exception as e:
                print(f"WARNING: no CCCL headers in the tarball and could not fetch them ({type(e).__name__}: {e}); kernels MLX compiles at run time may fail")
        vprint("headers for MLX's run-time kernels:", ", ".join(sorted(x.name for x in inc.iterdir())), "(in", str(inc) + ")")
        # 3. Does it run, on the GPU? One real operation, not --help.
        r = subprocess.run([str(cli), "device"], capture_output=True, text=True, timeout=900, env=dict(os.environ, LD_LIBRARY_PATH=ld))
        if r.returncode != 0 or "mlx device: gpu" not in r.stdout:
            cuda_diagnostics(cli, ld, probe=r)
            if r.returncode != 0:
                last = (r.stderr.strip().splitlines() or ["(no output)"])[-1]
                raise RuntimeError(f"`shears-cli device` failed: {exit_description(r.returncode)}; last line of its stderr: {last}")
            raise RuntimeError("the CUDA build started but MLX did not choose the GPU")
        vprint(r.stderr.strip()); vprint(r.stdout.strip())
        os.environ["LD_LIBRARY_PATH"] = ld
        # MLX keeps the kernels it compiled (as cubin, keyed by module name only) in files, and reads them back on the next run. The first
        # use of each shape costs a compile of 1 to 5 minutes on Colab's 2 vCPUs (first T4 run: 311, 155, 103 s per step, then seconds). One
        # directory per compute capability and build, because the files do not record either. Point SHEARS_JIT_CACHE at a folder that outlives the runtime (a mounted
        # Drive) to keep them across sessions, or SHEARS_JIT_CACHE_URL at a tarball made by the last cell of this notebook.
        jit_cache = os.environ.get("SHEARS_JIT_CACHE") or str(WORK / "jit-cache")
        build = hashlib.sha256((cli.parent.parent / "BUILD_INFO.txt").read_bytes()).hexdigest()[:8]     # a new build does not read an old build's kernels
        os.environ["MLX_PTX_CACHE_DIR"] = os.path.join(jit_cache, f"sm{str(GPU['cc']).replace('.', '')}-{build}")
        vprint("MLX run-time kernel cache:", os.environ["MLX_PTX_CACHE_DIR"])
        cache_url = os.environ.get("SHEARS_JIT_CACHE_URL", "")
        if cache_url and not any(pathlib.Path(os.environ["MLX_PTX_CACHE_DIR"]).glob("*.ptx")):
            name = os.path.basename(os.environ["MLX_PTX_CACHE_DIR"])
            try:
                if name not in cache_url:
                    raise ValueError(f"the cache file name must contain {name} (this GPU's compute capability and this build); a cache from another GPU or build would be loaded wrongly")
                pathlib.Path(os.environ["MLX_PTX_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
                fetch(cache_url, WORK / "jit-cache-download.tar.gz")
                with tarfile.open(WORK / "jit-cache-download.tar.gz") as tf:
                    tf.extractall(os.environ["MLX_PTX_CACHE_DIR"], filter="data") if hasattr(tarfile, "data_filter") else tf.extractall(os.environ["MLX_PTX_CACHE_DIR"])
                print("restored the compiled GPU kernels from", cache_url)
            except Exception as e:
                print(f"WARNING: could not restore the kernel cache ({type(e).__name__}: {e}); kernels will compile on first use")
        PHASES.append(("CUDA shears-cli (download, verify, unpack, libraries, run on GPU)", time.time() - t1))
        return cli

    cli = None
    t0 = time.time()
    try:
        install_gxx()
        cli = try_cuda_build()
        CUDA_OK = True
        vprint("using the CUDA shears-cli:", cli)
    except Exception as e:
        CUDA_REASON = f"{type(e).__name__}: {e}"
        PHASES.append(("CUDA shears-cli (failed or skipped)", time.time() - t0))
        print("\n*** Not using the CUDA build:", CUDA_REASON)
        print("*** Falling back to the CPU build.\n")

    # ---------------------------------------------------------------------------------------------
    # The CPU build.
    # ---------------------------------------------------------------------------------------------
    if cli is None:
        t0 = time.time()
        try:
            cli = fetch_and_unpack(PREBUILT_URL, PREBUILT_SHA256, WORK / "prebuilt")
            # Does this binary run on this system at all (glibc, libstdc++, the bundled libraries)?
            r = subprocess.run([str(cli), "--help"], capture_output=True, text=True, timeout=120)
            if r.returncode != 0:
                raise RuntimeError(f"shears-cli --help exited {r.returncode}: {(r.stderr or r.stdout).strip()[:500]}")
            PHASES.append(("prebuilt shears-cli (download, verify, unpack, test)", time.time() - t0))
            install_gxx()
        except Exception as e:
            PHASES.append(("prebuilt shears-cli (failed)", time.time() - t0))
            raise RuntimeError(f"""Could not set up the prebuilt shears-cli: {type(e).__name__}: {e}

Nothing was built. Check the network and PREBUILT_URL in section 1, then rerun this cell. To build shears-cli by hand instead, see
https://github.com/a1cd/Shears/blob/main/docs/linux.md (the notebook has no build-from-source cell).""") from None
    SHEARS_CLI = cli
    add_to_path(cli.parent)
    PREBUILT_OK = True
    print("shears-cli ready:", "CUDA build" if CUDA_OK else "CPU build", "(prebuilt, checksum verified)")
    vprint("path:", cli)
else:
    print("not Linux: skipping")

### 2c. Which device MLX uses

`shears-cli device` does one small matrix product on MLX's default device. The next cell reports the answer in one line: the GPU it uses, or that it is on the CPU. (`VERBOSE = True` also prints the command's own output.)

In [ ]:
import re
r = subprocess.run([str(SHEARS_CLI), "device"], capture_output=True, text=True, timeout=900)
if r.stderr.strip() and VERBOSE:
    print(r.stderr.strip())
m = re.search(r"mlx device: (\w+)", r.stdout)
if m and r.returncode == 0:
    DEVICE = m.group(1)
    vprint(r.stdout.strip())
elif IS_LINUX and not CUDA_OK:
    DEVICE = "cpu"
    vprint("mlx device: cpu   (this prebuilt predates `shears-cli device`, or it printed nothing; it is the CPU build, so MLX uses the CPU)")
else:
    raise RuntimeError(f"`shears-cli device` failed ({exit_description(r.returncode)}):\n{(r.stdout + r.stderr).strip()}")
if CUDA_REASON:
    print("(CUDA build not used:", CUDA_REASON + ")")
if DEVICE == "gpu":
    print("GPU: " + (GPU["name"] + " (CUDA)" if CUDA_OK else "Apple Silicon (Metal)") + ": Shears is using it")
else:
    print("No GPU in use: MLX is on the CPU" + ("" if MODEL == "tiny" else " (too slow for the real model)"))

## 3. Prune and heal

**`MODEL = "tiny"`** (the cells up to section 3g): `shears-cli sample` writes a tiny random Gemma 4 MoE (4 layers, 8 experts each) and four short rows to heal on. `Examples/tiny-config.json` is its config. Then four commands, each reading the same config and safe to rerun: `select`, `prepass`, `heal`, `export`. With `MODEL = "gemma"` these cells say they are skipping, and section 3g does the real thing.

In [ ]:
SETUP_MODE = "xcodebuild" if IS_MAC else (("prebuilt-cuda" if CUDA_OK else "prebuilt") if PREBUILT_OK else "unknown")
SETUP_SECONDS = sum(s for _, s in PHASES)
print(f"setup: {SETUP_MODE}, {SETUP_SECONDS:.1f} s")

In [ ]:
if MODEL == "tiny":
    TINY = WORK / "tiny"
    shutil.rmtree(TINY, ignore_errors=True)
    sh(f'"{SHEARS_CLI}" sample "{TINY}"', "sample workspace")
    # Trim the sample config to a smoke test: 3 heal steps per stage (the sample has 6, 6 and 5).
    cfg = json.load(open(SRC / "Examples/tiny-config.json"))
    for st in cfg["stages"]:
        st["steps"] = 3
    cfg["stages"][2]["plan"] = "freerun@0,end@2"   # stage C still reaches its second phase
    json.dump(cfg, open(TINY / "config.json", "w"), indent=2)
    print(sorted(p.name for p in TINY.iterdir()))
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`select`** keeps 4 of the 8 experts in each layer: the ones whose removal changes the layer's output the least on the rows.

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" select --config config.json', "shears-cli select")
    for i, kept in enumerate(json.load(open(TINY / "out/keep.json"))["keep"]):
        print(f"layer {i}: keeps experts {kept} of 0-7")
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`prepass`** runs the original (unpruned) model once over the rows and stores what healing needs, so the big model never has to run again. This is the "teacher".

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" prepass --config config.json', "shears-cli prepass")
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`heal`** trains small adapters (for the experts, the dense layers and the router) so the pruned model's outputs move back toward the teacher's. Stages A and B heal one block of layers at a time; stage C runs over the whole pruned model. A few toy steps on a random model say nothing about quality, and the loss need not fall: this only shows the loop running.

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" heal --config config.json', "shears-cli heal")
    for stage in "ABC":
        rows = [json.loads(l) for l in open(TINY / f"out/stage-{stage}/steps.jsonl")]
        print(f"stage {stage}: {len(rows)} steps, loss {rows[0]['loss']:.3f} -> {rows[-1]['loss']:.3f}")
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`export`** merges the adapters into the weights and writes only the kept experts, in Hugging Face layout (bfloat16). The same folder feeds `shears-cli gguf` (a GGUF file for llama.cpp) and `shears-serve` (an OpenAI-style server).

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" export --config config.json', "shears-cli export")
    EXPORT = TINY / "out/export"
    for p in sorted(EXPORT.iterdir()):
        print(f"{p.stat().st_size:>9,}  {p.name}")
    cfg = json.load(open(EXPORT / "config.json"))
    print("\nexperts per layer in the export:", cfg["num_experts"], "(the original had 8); experts used per token:", cfg["top_k_experts"])
else:
    print('MODEL is gemma: skipping the tiny pipeline')

## 3g. The real model: `MODEL = "gemma"`

Skipped with `MODEL = "tiny"`. In order:

1. **Check** the disk and the device (the notebook stops before the download if MLX is on the CPU, see `ALLOW_GEMMA_ON_CPU`), then **download** `mlx-community/gemma-4-26B-A4B-it-qat-4bit` (about 16 GB; Google's 4-bit QAT checkpoint, Apache 2.0).
2. **Choose the rows to heal on**: 32 English sentences from the WMT24++ test set, each with its human Japanese translation, written as a translator conversation with the system prompt in `Notebooks/data/translate-system-prompt-ja.txt`. Healing makes the pruned model's outputs match the original model's on whatever text it is shown, so these sentences are only practice material. The 200 sentences used for the final check are never among them.
3. **`shears-cli run`**: for each block of 6 layers it chooses the 8 experts to keep in each layer (of 128), then trains small adapters on that block for a few steps. Layers are streamed from disk one at a time, so the model never has to fit in memory. Every block is healed because the export needs adapters for every layer.
4. **`shears-cli export`** merges the adapters and writes only the kept experts, about 3.8 billion parameters.

This is a short run to show the whole pipeline, not a recipe for a good model: real quality needs far more rows and steps.

In [ ]:
# 3g.1  Python packages, then the plan: the size of the download, the free disk, the device.
GEMMA_DIR = WORK / "gemma"
MODEL_DIR = GEMMA_DIR / "model"
if MODEL == "gemma":
    need_pkgs = [f"transformers=={TRANSFORMERS_VERSION}", "huggingface_hub"]
    try:
        import torch
    except ImportError:
        need_pkgs.append("torch")
    step("Installing the Python packages (transformers)", "40 seconds")
    sh(f'"{sys.executable}" -m pip install -q --disable-pip-version-check {" ".join(need_pkgs)}', "pip install transformers", quiet=not VERBOSE)
    import transformers
    assert transformers.__version__ == TRANSFORMERS_VERSION, "restart the runtime so the pinned transformers is imported, then run again from the top"
    transformers.utils.logging.disable_progress_bar()
    from huggingface_hub import HfApi

    PATTERNS = ["*.json", "*.safetensors", "*.jinja", "tokenizer*"]
    try:
        info = HfApi().model_info(GEMMA_REPO, files_metadata=True)
        import fnmatch
        files = [s for s in info.siblings if any(fnmatch.fnmatch(s.rfilename, p) for p in PATTERNS) and not s.rfilename.startswith(".")]
        DOWNLOAD_BYTES = sum(s.size or 0 for s in files)
        size_source = "Hugging Face file list"
    except Exception as e:
        DOWNLOAD_BYTES = 15_600_000_000
        size_source = f"recorded 2026-10-22 (could not reach the file list: {type(e).__name__})"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    have = sum(f.stat().st_size for f in MODEL_DIR.rglob("*") if f.is_file())
    remaining = max(DOWNLOAD_BYTES - have, 0)
    # After the download: the exported model (about 3.8B parameters, 7.6 GB in bf16, by calculation), the healing workspace and slack.
    WORKSPACE_BYTES = 14_000_000_000
    free = shutil.disk_usage(WORK).free
    vprint(f"download : {DOWNLOAD_BYTES / 1e9:.1f} GB ({size_source}); already here: {have / 1e9:.1f} GB")
    vprint(f"time     : at 50 to 150 MB/s that is {remaining / 150e6 / 60:.0f} to {remaining / 50e6 / 60:.0f} minutes (Colab's speed varies)")
    vprint(f"disk     : {free / 1e9:.1f} GB free in {WORK}; this run needs about {(remaining + WORKSPACE_BYTES) / 1e9:.0f} GB more (download + export + workspace)")
    if free < remaining + WORKSPACE_BYTES:
        raise SystemExit(
            f"STOP: not enough disk. Need about {(remaining + WORKSPACE_BYTES) / 1e9:.0f} GB free, have {free / 1e9:.1f} GB. "
            "Delete files, pick a runtime with more disk, or set MODEL = \"tiny\" in the first code cell.")
    vprint(f"device   : MLX uses the {DEVICE.upper()}")
    if DEVICE == "cpu" and not IS_MAC and not ALLOW_GEMMA_ON_CPU:
        raise SystemExit(
            "STOP: MLX is on the CPU" + (f" ({CUDA_REASON})" if CUDA_REASON else "") + ". The real model on a CPU would take many hours at the very least, "
            "so the notebook stops before downloading 16 GB. On Colab pick a GPU runtime (Runtime, Change runtime type) and run again from the top. "
            "To go on anyway set ALLOW_GEMMA_ON_CPU = True in section 1, or set MODEL = \"tiny\".")
    print(f"Checks passed: {free / 1e9:.0f} GB of disk free (about {(remaining + WORKSPACE_BYTES) / 1e9:.0f} GB needed) and MLX is using the {DEVICE.upper()}.")
else:
    print("MODEL is tiny: skipping")

In [ ]:
# 3g.2  The config. Change the numbers here. Paths are relative to this file's folder. It is checked by `shears-cli` before anything is downloaded.
def memory_budget_gb():
    """A ceiling for `shears-cli run`: it refuses up front, with the amount, if the run needs more."""
    if os.environ.get("SHEARS_MEMORY_GB"):
        return float(os.environ["SHEARS_MEMORY_GB"])
    if IS_MAC:
        mem = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout) / 2**30
        return round(mem * 0.6, 1)
    if DEVICE == "gpu" and GPU:
        return round(GPU["mem_mib"] / 1024 * 0.85, 1)
    meminfo = dict(l.split(":", 1) for l in open("/proc/meminfo"))
    return round(int(meminfo["MemAvailable"].split()[0]) / 2**20 * 0.7, 1)

# T4 and older Turing parts have no bfloat16 arithmetic: use float32 for the maths there, and on the CPU. Layers are streamed one at a time,
# so that costs little memory. Ampere and newer, and Macs, use bfloat16. The export is bfloat16 either way.
COMPUTE_DTYPE = "float32" if (DEVICE == "cpu" and not IS_MAC) or (GPU and not IS_MAC and GPU["cc"] < 8.0) else "bfloat16"
BUDGET_GB = memory_budget_gb()
GEMMA_CONFIG = {
    "model": "model", "dtype": COMPUTE_DTYPE, "rows": "rows.jsonl", "keep": "out/keep.json",
    "order": {"kind": "shuffled", "seed": 9}, "checkpointDir": "checkpoints", "outputDir": "out", "stopFile": "STOP",
    "seed": 1, "progressEvery": 1,
    "memory": {"budgetGB": BUDGET_GB},
    "adapters": {"expertRank": 8, "denseRank": 8, "routerRank": 4},
    # 8 of 128 experts per layer. importanceCandidates and timeLimitPerLayer keep the search short; the recipe's defaults are slower.
    "select": {"keep": 8, "importanceCandidates": 32, "swapRounds": 1, "timeLimitPerLayer": 60},
    "pipeline": {"steps": GEMMA_STEPS_PER_BLOCK, "batchSteps": 5, "mix": "both", "rowsPerStep": 1, "sharedPrefix": "auto",
                 "learningRate": 0.001, "learningRateFloor": 0.0001, "warmup": 2, "checkpointEvery": 5,
                 "keepIntermediates": "lowMemory", "maxBatchTokens": 4096, "maxBatchRows": 8},   # 4096: rows are about 400 tokens long, so 5 of them fit one walk
    "export": {"dtype": "bfloat16"},
}
# The export step gets the same config without "dtype", so layers are read as stored. It streams one layer at a time (needs only out/keep.json and
# the adapters), so it fits Colab's RAM; it can be rerun on its own when out/ is still there.
EXPORT_CONFIG = {k: v for k, v in GEMMA_CONFIG.items() if k != "dtype"}
if MODEL == "gemma":
    GEMMA_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(GEMMA_CONFIG, open(GEMMA_DIR / "config.json", "w"), indent=2)
    json.dump(EXPORT_CONFIG, open(GEMMA_DIR / "config-export.json", "w"), indent=2)
    # Does shears-cli accept the config? Run it before the 16 GB download. The model is not there yet, so it must fail, and only at that point.
    for name in ("config.json", "config-export.json"):
        cmd = "run" if name == "config.json" else "export"
        r = subprocess.run([str(SHEARS_CLI), cmd, "--config", name], cwd=GEMMA_DIR, capture_output=True, text=True, timeout=600)
        out = (r.stdout + r.stderr).strip()
        if r.returncode == 0 or any(w in out for w in ("shears-cli: config:", "DecodingError", "keyNotFound", "typeMismatch", "valueNotFound", "dataCorrupted")):
            raise RuntimeError(f"shears-cli does not accept {name} (exit {r.returncode}):\n{out[-1500:]}")
        vprint(f"{name}: accepted (shears-cli then stopped at the missing model, as expected: {(out.splitlines() or [''])[-1][:120]})")
    print(f"Plan: {GEMMA_STEPS_PER_BLOCK} healing steps in each of 5 blocks of 6 layers, keeping 8 of the 128 experts in every layer. Settings checked.")
    vprint(f"compute dtype {COMPUTE_DTYPE}; memory budget {BUDGET_GB} GB (a ceiling: the run is refused up front, with the amount, if it needs more)")
else:
    print("MODEL is tiny: skipping")

In [ ]:
# 3g.3  Download the model (resumable: rerun if it stops).
if MODEL == "gemma":
    from huggingface_hub import snapshot_download
    step(f"Downloading the model, {DOWNLOAD_BYTES / 1e9:.1f} GB (the bar below shows progress; it resumes if you run the cell again)", "3 minutes")
    t0 = time.time()
    import threading
    def bytes_on_disk():
        # Finished files plus the partial ones Hugging Face writes under .cache while downloading.
        return sum(f.stat().st_size for f in MODEL_DIR.rglob("*") if f.is_file())
    start_bytes = bytes_on_disk()
    bar = Progress(DOWNLOAD_BYTES, "model download", start=start_bytes)
    stop = threading.Event()
    def watch():
        while not stop.wait(1.0):
            bar.update(bytes_on_disk(), force=True)
    watcher = threading.Thread(target=watch, daemon=True)
    watcher.start()
    try:
        snapshot_download(repo_id=GEMMA_REPO, local_dir=str(MODEL_DIR), allow_patterns=PATTERNS)
    finally:
        stop.set()
        watcher.join()
        bar.done(bytes_on_disk())
    PHASES.append(("download mlx-community/gemma-4-26B-A4B-it-qat-4bit", time.time() - t0))
    shards = sorted(MODEL_DIR.glob("*.safetensors"))
    print(f"{len(shards)} safetensors files, {sum(p.stat().st_size for p in shards) / 1e9:.1f} GB, in {MODEL_DIR}")
    assert (MODEL_DIR / "config.json").exists() and shards, "the download is incomplete: run this cell again"
else:
    print("MODEL is tiny: skipping")

In [ ]:
# 3g.4  The training rows. English sentences from WMT24++ (en-ja_JP), each with its human Japanese translation, written in the model's own chat
# template: the translator system prompt, the English as the user message, the Japanese as the reply. answerStart marks where the reply begins.
# (Healing pulls the pruned model's outputs toward the original model's on whatever text it is shown, so the human Japanese is only the text the
# model reads here; the original model stays the target.) The 200 sentences used for the final check are left out.
if MODEL == "gemma":
    import importlib.util, random
    from transformers import AutoTokenizer
    from huggingface_hub import hf_hub_download

    for p in (SRC / SYSTEM_PROMPT_FILE, SRC / GEMMA_DATA, SRC / "Notebooks/data/make_gemma_en_ja.py"):
        assert p.exists(), (f"{p} is missing. These files are in Notebooks/data/ of the Shears repository. If the notebook cloned the sources itself, "
                            f"the cloned branch ({SHEARS_REF}) does not have them yet: set SHEARS_REF in section 1 to a branch that does.")
    SYSTEM_PROMPT = (SRC / SYSTEM_PROMPT_FILE).read_text(encoding="utf-8")
    HELDOUT = [json.loads(l) for l in open(SRC / GEMMA_DATA, encoding="utf-8")]
    held_keys = {(r["document_id"], r["segment_id"]) for r in HELDOUT}
    # The same filter that picked those 200 (no URLs, no canary or bad-source rows, nothing offensive or political) picks the training sentences.
    spec = importlib.util.spec_from_file_location("gemma_en_ja", SRC / "Notebooks/data/make_gemma_en_ja.py")
    gd = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(gd)
    wmt_path = hf_hub_download(gd.WMT_REPO, gd.WMT_FILE, repo_type="dataset", revision=gd.WMT_REVISION)
    pool = [r for r in (json.loads(l) for l in open(wmt_path, encoding="utf-8")) if gd.usable(r) and (r["document_id"], r["segment_id"]) not in held_keys]
    by_domain = {}
    for r in sorted(pool, key=lambda r: (r["document_id"], r["segment_id"])):
        by_domain.setdefault(r["domain"], []).append(r)
    rng = random.Random(5)
    for v in by_domain.values():
        rng.shuffle(v)
    TRAIN, per_doc, k = [], {}, 0
    while len(TRAIN) < TRAIN_ROWS and any(by_domain.values()):       # one from each domain in turn, at most 2 per document
        d = sorted(by_domain)[k % len(by_domain)]
        k += 1
        while by_domain[d]:
            r = by_domain[d].pop()
            if per_doc.get(r["document_id"], 0) < 2:
                per_doc[r["document_id"]] = per_doc.get(r["document_id"], 0) + 1
                TRAIN.append(r)
                break
        if not by_domain[d]:
            del by_domain[d]
    assert not ({(r["document_id"], r["segment_id"]) for r in TRAIN} & held_keys)

    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    END_ID = tok.convert_tokens_to_ids("<turn|>")                                   # the end-of-turn token the model writes after its reply
    assert tok.convert_ids_to_tokens(END_ID) == "<turn|>", "this tokenizer has no <turn|> token: is it a Gemma 4 tokenizer?"

    def ids_of(tokenizer, messages, **kw):
        out = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=True, **kw)["input_ids"]
        return list(out[0]) if out and isinstance(out[0], (list, tuple)) else list(out)

    def prompt_ids(tokenizer, english):
        """The text the model reads before it answers: system prompt, the English, and the generation prompt. Thinking is off, so (in Gemma 4's own
        template) an empty thought channel ends the prompt. Training and every later check build the prompt with this one function."""
        return ids_of(tokenizer, [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": english}],
                      add_generation_prompt=True, enable_thinking=False)

    rows = []
    for n, r in enumerate(TRAIN):
        prompt = prompt_ids(tok, r["source"])
        answer = tok(r["target"], add_special_tokens=False)["input_ids"] + [END_ID]
        rows.append({"key": f"w{n:02d}", "tokens": [int(t) for t in prompt + answer], "answerStart": len(prompt)})
    with open(GEMMA_DIR / "rows.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    lens = [len(r["tokens"]) for r in rows]
    assert max(lens) < 1500, "a row is longer than expected"
    print(f"{len(rows)} training rows, {min(lens)} to {max(lens)} tokens each. {len(HELDOUT)} other sentences, with Gemma's own translations, are kept for the final check.")
    if VERBOSE:
        print("first row decodes to:", repr(tok.decode(rows[0]["tokens"])[-300:]))
        print("domains:", {d: sum(1 for r in TRAIN if r["domain"] == d) for d in sorted({r["domain"] for r in TRAIN})})
else:
    print("MODEL is tiny: skipping")

**`run`** does the work, block by block, and can be rerun after a stop or a disconnect (it continues from its checkpoints). It prints one status line per phase that updates in place; the full output is saved to `run.log` (or set `VERBOSE = True` to see every line).

In [ ]:
if MODEL == "gemma":
    step("Pruning the model and healing it, one block of 6 layers at a time (5 blocks)",
         "18 minutes in all, timed with shorter training rows than these: block 1 takes about 12 because it also compiles GPU kernels; blocks 2 to 5 about 1 minute each")
    RUN_LOG = GEMMA_DIR / "run.log"
    run_status = None
    if not VERBOSE:
        run_status = Status("Block 1/5: running the original model on the rows (the first block also compiles GPU kernels: several minutes with no news)")
    def on_run_line(line):
        """Turns shears-cli's lines into the one status line. Everything else goes to run.log only."""
        m = re.match(r"run: block (\d+)/(\d+): selected layer (\d+)", line)
        if m:
            run_status.set(f"Block {m[1]}/{m[2]}: choosing which experts to keep (layer {m[3]})")
        m = re.match(r"\[pipeline block (\d+)/(\d+)\] step (\d+)/(\d+)\s+loss ([\d.]+)", line)
        if m:
            run_status.set(f"Block {m[1]}/{m[2]}: healing, step {m[3]}/{m[4]} (loss {float(m[5]):.2f})")
        m = re.match(r"run: block (\d+)/(\d+) trained", line)
        if m:
            nxt = int(m[1]) + 1
            run_status.set(f"Block {m[1]}/{m[2]} done" + (f"; block {nxt}/{m[2]}: running the original model on the rows" if nxt <= int(m[2]) else "; writing the adapters"))
    try:
        sh(f'cd "{GEMMA_DIR}" && "{SHEARS_CLI}" run --config config.json', "shears-cli run (select and heal, every block)",
           on_line=on_run_line, log=RUN_LOG)
    except BaseException:
        if run_status:
            run_status.done(run_status.text + "   [stopped]")
        raise
    keep = json.load(open(GEMMA_DIR / "out/keep.json"))["keep"]
    if run_status:
        run_status.done(f"Pruned and healed: kept {len(keep[0])} of the 128 experts in each of {len(keep)} layers.")
    print(f"(the full log of this step: {RUN_LOG})")
    vprint(f"layer 0 keeps experts {keep[0]}")
    for p in sorted((GEMMA_DIR / "out").glob("**/steps.jsonl")):
        by_block = {}
        for l in open(p):
            r_ = json.loads(l)
            by_block.setdefault(r_.get("block", 0), []).append(r_["loss"])
        for b_, ls in sorted(by_block.items()):
            print(f"block {b_ + 1}: {len(ls)} steps, loss {ls[0]:.3f} -> {ls[-1]:.3f}")
else:
    print("MODEL is tiny: skipping")

**`export`** merges the adapters into the kept experts and writes the pruned model in Hugging Face layout (bfloat16). Same status line; full output in `export.log`.

In [ ]:
if MODEL == "gemma":
    step("Exporting: merging the healed adapters into the 8 kept experts of each of the 30 layers, one layer at a time", "3 minutes")
    EXPORT_LOG = GEMMA_DIR / "export.log"
    export_status = None if VERBOSE else Status("Exporting: starting")
    def on_export_line(line):
        m = re.match(r"export: layer (\d+)/(\d+) merged", line)
        if m:
            export_status.set(f"Exporting: layer {m[1]}/{m[2]}")
    try:
        sh(f'cd "{GEMMA_DIR}" && "{SHEARS_CLI}" export --config config-export.json', "shears-cli export", on_line=on_export_line, log=EXPORT_LOG)
    except BaseException:
        if export_status:
            export_status.done(export_status.text + "   [stopped]")
        raise
    GEMMA_EXPORT = GEMMA_DIR / "out/export"
    files = sorted(GEMMA_EXPORT.iterdir())
    if export_status:
        export_status.done(f"Exported {len(files)} files, {sum(p.stat().st_size for p in files) / 1e9:.1f} GB, to {GEMMA_EXPORT}")
    print(f"(the full log of this step: {EXPORT_LOG})")
    for p in files:
        vprint(f"{p.stat().st_size:>14,}  {p.name}")
else:
    print("MODEL is tiny: skipping")

## 4. Use the pruned model

The export is an ordinary Hugging Face checkpoint, so `AutoModelForCausalLM` loads it. transformers is pinned to a version that has Gemma 4. (On Colab, if `import transformers` ran earlier in this session, restart the runtime after the install cell so the pinned version is the one imported.)

- **`tiny`**: generates a few tokens of gibberish. That is expected for a random model.
- **`gemma`**: loads the pruned model (about 8 GB in bf16) and compares it with the real Gemma on the 50 held-out sentences, `CHECK_ROWS` in section 1 (the saved file has 200). Two checks:
  1. **Agreement.** The pruned model reads the system prompt, the English and Gemma's translation, and for each token Gemma wrote we ask: was that the pruned model's top choice, or among its 5 best? This needs no generation and is the clearest measure.
  2. **Side by side.** English, Gemma's saved translation and the pruned model's own translation for the first 5 sentences.

The saved Gemma translations come from `google/gemma-4-26b-a4b-it` on OpenRouter (full precision, temperature 0, no thinking, the same system prompt), made on 2026-10-22 with `Notebooks/data/make_gemma_en_ja.py`. No key is needed to read them.

In [ ]:
need = [f"transformers=={TRANSFORMERS_VERSION}"]
try:
    import torch
except ImportError:
    need.append("torch")
sh(f'"{sys.executable}" -m pip install -q --disable-pip-version-check {" ".join(need)}', "pip install transformers", quiet=not VERBOSE)
import transformers
print("transformers", transformers.__version__)
assert transformers.__version__ == TRANSFORMERS_VERSION, "restart the runtime so the pinned transformers is imported"

In [ ]:
if MODEL == "tiny":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    transformers.utils.logging.disable_progress_bar()

    tokenizer = AutoTokenizer.from_pretrained(EXPORT)
    model = AutoModelForCausalLM.from_pretrained(EXPORT, dtype=torch.bfloat16)   # the export is bf16
    model.eval()
    print(type(model).__name__, f"| {sum(p.numel() for p in model.parameters()):,} parameters | dtype {next(model.parameters()).dtype}")

    # --- Change the prompt here. The tokenizer is a toy (lower-case letters and bytes), so use short plain text.
    messages = [{"role": "user", "content": "hello"}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)

    # --- Change sampling here. A tiny RANDOM model writes gibberish whatever you do; this only shows the model runs.
    # For varied output set do_sample=True and temperature / top_k / top_p. Greedy decoding is repeatable.
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    print("prompt   :", repr(tokenizer.decode(inputs["input_ids"][0])))
    print("generated:", repr(tokenizer.decode(new_tokens)))
    print("token ids:", new_tokens.tolist())
    GENERATED_TEXT = tokenizer.decode(new_tokens)
else:
    import torch, gc
    from transformers import AutoTokenizer, AutoModelForCausalLM
    transformers.utils.logging.disable_progress_bar()
    if not VERBOSE:
        transformers.utils.logging.set_verbosity_error()   # hides transformers' long table of weights it could not place; the loader below reports anything that matters
    gc.collect()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(GEMMA_EXPORT)
    END_ID = tokenizer.convert_tokens_to_ids("<turn|>")
    CHECK, SHOW = HELDOUT[:CHECK_ROWS], HELDOUT[:SHOW_ROWS]
    n_check = len(CHECK)

    def load_pruned(dtype):
        model, info = AutoModelForCausalLM.from_pretrained(GEMMA_EXPORT, dtype=dtype, output_loading_info=True)
        model = model.to(device).eval()
        # A text-only export has no vision or audio weights; those are expected to be missing. Anything else missing would be a real problem.
        missing = [k for k in info.get("missing_keys", []) if not re.search(r"vision|audio|multi_modal|embed_vision|embed_audio", k)]
        print(f"Loaded the pruned model on the {device.upper()}: {sum(p.numel() for p in model.parameters()) / 1e9:.1f} billion parameters, {next(model.parameters()).dtype}.")
        if missing:
            print(f"WARNING: {len(missing)} weights of the language model are missing from the export, for example {missing[:3]}. The output below is not trustworthy.")
        vprint(type(model).__name__, "| missing keys:", len(info.get("missing_keys", [])), "| unexpected keys:", len(info.get("unexpected_keys", [])))
        return model

    # Stop at the end of the reply (<turn|>), as well as at whatever the model's own settings say.
    def stop_ids(model):
        g = model.generation_config.eos_token_id
        ids = set([g] if isinstance(g, int) else (g or [])) | {END_ID, tokenizer.eos_token_id}
        return sorted(i for i in ids if isinstance(i, int) and i >= 0)

    def translate(model, english):
        """The pruned model's own greedy translation (the same system prompt and prompt format as in training)."""
        p = prompt_ids(tokenizer, english)
        x = torch.tensor([p], device=device)
        n_english = len(tokenizer(english, add_special_tokens=False)["input_ids"])
        with torch.no_grad():
            out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=max(64, min(320, 3 * n_english + 32)), do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, eos_token_id=stop_ids(model))
        new = out[0, len(p):]
        return new.tolist(), tokenizer.decode(new, skip_special_tokens=True).strip()

    def agreement(model, row):
        """Teacher forcing: show the pruned model the system prompt, the English and Gemma's translation, and count at how many of Gemma's tokens the
        pruned model's top choice (top-1) or one of its 5 best choices (top-5) is the token Gemma actually wrote. Includes the end-of-reply token."""
        p = prompt_ids(tokenizer, row["english"])
        a = tokenizer(row["gemma_ja"], add_special_tokens=False)["input_ids"] + [END_ID]
        x = torch.tensor([p + a], device=device)
        with torch.no_grad():
            logits = model(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False).logits[0, len(p) - 1 : -1]
        top5 = logits.topk(5, dim=-1).indices
        t = torch.tensor(a, device=device)
        return int((top5[:, 0] == t).sum()), int((top5 == t[:, None]).any(dim=1).sum()), len(a)

    def run_all(dtype):
        global PRUNED
        PRUNED = load_pruned(dtype)
        st = Status(f"Comparing with Gemma: sentence 0/{n_check}")
        agree = []
        for k, row in enumerate(CHECK):
            agree.append(agreement(PRUNED, row))
            st.set(f"Comparing with Gemma: sentence {k + 1}/{n_check}")
        st.done(f"Compared with Gemma on {n_check} sentences.")
        st = Status(f"Translating {len(SHOW)} sentences with the pruned model: 0/{len(SHOW)}")
        shown = []
        for k, row in enumerate(SHOW):
            shown.append(translate(PRUNED, row["english"]))
            st.set(f"Translating {len(SHOW)} sentences with the pruned model: {k + 1}/{len(SHOW)}")
        st.done(f"Translated {len(SHOW)} sentences with the pruned model.")
        return agree, shown

    step(f"Loading the pruned model, comparing it with Gemma's translations of {n_check} held-out sentences and translating {len(SHOW)} of them "
         "(on the Colab T4, loading plus one short translation took 1.5 minutes when measured; the comparison on top of that has not been timed)")
    t0 = time.time()
    if device == "cuda" and torch.cuda.get_device_capability()[0] < 8:
        print("note: this GPU has no bfloat16 hardware (compute capability < 8); torch emulates it, which works but is slow")
    try:
        agree, shown = run_all(torch.bfloat16)               # the export is bf16
    except Exception as e:
        if device != "cuda":
            raise
        print(f"bfloat16 failed on this GPU ({type(e).__name__}: {str(e)[:200]}); trying float16. Gemma can overflow in float16, so read the results with care.")
        PRUNED = None
        gc.collect(); torch.cuda.empty_cache()
        agree, shown = run_all(torch.float16)

    tot = sum(n for _, _, n in agree)
    top1, top5 = sum(a for a, _, _ in agree) / tot, sum(b for _, b, _ in agree) / tot
    per = sorted(a / n for a, _, n in agree)
    print(f"\nCompared with Gemma's own translations of {n_check} sentences ({tot} tokens). At each token the pruned model's top choice is the token Gemma wrote "
          f"{top1:.0%} of the time, and that token is among its 5 best choices {top5:.0%} of the time. Per sentence the top-choice rate runs from {per[0]:.0%} to {per[-1]:.0%} (median {per[len(per) // 2]:.0%}).")
    print("Gemma's translations came from the full-precision model (through OpenRouter); the model Shears healed against is the 4-bit checkpoint, so a small difference is expected even for a perfect copy.")
    show_table(["English", "tokens", "top choice = Gemma's token", "Gemma's token in top 5"],
               [(r["english"][:90] + ("..." if len(r["english"]) > 90 else ""), n, f"{a / n:.0%}", f"{b / n:.0%}") for r, (a, b, n) in zip(CHECK, agree)],
               caption="Per sentence (scroll for all)", max_height=300)
    show_table(["English", "Gemma 4 26B-A4B (saved translation)", "Pruned and healed model (Shears)"],
               [(r["english"], r["gemma_ja"], text) for r, (_, text) in zip(SHOW, shown)], caption="Side by side")
    new_tokens = [t for toks, _ in shown for t in toks]
    GENERATED_TEXT = "\n".join(text for _, text in shown)
    PHASES.append(("transformers: load the pruned Gemma, compare and translate", time.time() - t0))
print(f"[{MODEL}] done: generated {len(new_tokens)} tokens")

### 4a. Try your own sentence

Type English into `MY_ENGLISH` in the next cell and run it. The pruned model translates it. If you add your own Gemini API key (steps in the cell) it also shows real Gemma's translation, live, next to it.

In [ ]:
# 4a. Translate your own sentence with the pruned model. Change the text, then run only this cell (the model stays loaded).
MY_ENGLISH = "Can you tell me a joke about otters?"  # @param {type:"string"}

# Optional: also see what real Gemma 4 26B-A4B says, live, through Google's Gemini API with YOUR key. The key is read from Colab's Secrets (the key icon
# in the left bar: add a secret called GEMINI_API_KEY or GOOGLE_API_KEY and switch on Notebook access) or from an environment variable of the same name.
# It is only sent to generativelanguage.googleapis.com, in that one request header; it is never printed or saved. Without a key this cell shows the
# pruned model alone. (The 200 saved Gemma translations used above need no key.)
import urllib.error

def find_gemini_key():
    """(key, problem): the key from Colab Secrets or the environment, or None and what to tell the reader."""
    denied = None
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                k = userdata.get(name)
                if k:
                    return k, None
            except Exception as e:                      # secret not found, or notebook access not granted
                if "Access" in type(e).__name__ or "access" in str(e):
                    denied = name
    except ImportError:
        pass
    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        if os.environ.get(name):
            return os.environ[name], None
    if denied:
        return None, f"Colab has a secret called {denied} but this notebook may not read it: switch on Notebook access next to it in the key panel, then run this cell again."
    return None, ("To also see real Gemma's translation here: in Colab click the key icon in the left bar, add a secret GEMINI_API_KEY with your Gemini API key "
                  "and switch on Notebook access (elsewhere, set the environment variable GEMINI_API_KEY), then run this cell again.")

def gemini_translate(english, key):
    """Real Gemma 4 26B-A4B through the Gemini API: the same system prompt, temperature 0, thinking at its lowest. Returns (text, how)."""
    url = "https://generativelanguage.googleapis.com/v1beta/models/gemma-4-26b-a4b-it:generateContent"
    base = {"systemInstruction": {"parts": [{"text": SYSTEM_PROMPT}]}, "contents": [{"role": "user", "parts": [{"text": english}]}]}
    # The Gemma page of the Gemini API docs does not say which thinkingLevel Gemma 4 accepts. "minimal" is tried first; if the API refuses the
    # request (HTTP 400) it is sent once more without that setting (the model's chat template has thinking off unless asked).
    variants = [("thinking minimal", {**base, "generationConfig": {"temperature": 0, "thinkingConfig": {"thinkingLevel": "minimal"}}}),
                ("default thinking setting", {**base, "generationConfig": {"temperature": 0}})]
    for n, (how, body) in enumerate(variants):
        req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json", "x-goog-api-key": key})
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                resp = json.load(r)
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", "replace")[:300]
            if e.code == 400 and n < len(variants) - 1:
                continue
            raise RuntimeError(f"the Gemini API answered HTTP {e.code}: {detail}") from None
        except urllib.error.URLError as e:
            raise RuntimeError(f"could not reach the Gemini API: {e.reason}") from None
        parts = resp["candidates"][0]["content"]["parts"]
        return "".join(p.get("text", "") for p in parts if not p.get("thought")).strip(), how
    raise RuntimeError("the Gemini API refused every request")

if MODEL == "gemma" and globals().get("PRUNED") is not None:
    my_text = translate(PRUNED, MY_ENGLISH)[1]
    key, why = find_gemini_key()
    if key is None:
        print(why)
        show_table(["English", "Pruned and healed model (Shears)"], [(MY_ENGLISH, my_text)])
    else:
        try:
            live, how = gemini_translate(MY_ENGLISH, key)
            show_table(["English", f"Gemma 4 26B-A4B (live, your Gemini key; {how})", "Pruned and healed model (Shears)"], [(MY_ENGLISH, live, my_text)])
        except Exception as e:
            print("Gemma through the Gemini API did not work:", e)
            show_table(["English", "Pruned and healed model (Shears)"], [(MY_ENGLISH, my_text)])
else:
    print("This cell is for MODEL = gemma, after the cell above has run (the tiny model only writes gibberish).")

## 5. How long each phase took

In [ ]:
jit_cache_report(pack=True)
phase_table()
vprint(f"setup ({SETUP_MODE}): {SETUP_SECONDS:.1f} s")
json.dump({"model": MODEL, "generated": GENERATED_TEXT, "device": DEVICE, "mode": SETUP_MODE, "setup_seconds": SETUP_SECONDS, "total_seconds": sum(s for _, s in PHASES), "phases": PHASES},
          open(WORK / "phases.json", "w"), indent=1)   # read by CI

## Notes and caveats

**What was measured where.** The real target for Shears is MLX on an Apple Silicon Mac, and every number published for Shears (speed, memory, quality, energy) was measured there. Linux has no Metal, no zero-copy memory mapping and no unified memory, so do not compare the timings or losses here with the Mac numbers. A timing quoted in this notebook is from one Colab T4 run on 2026-10-22 (the full `gemma` pipeline ran end to end on it, with MLX on the GPU through the CUDA build); other GPUs, and the memory use of the CUDA build, have not been measured. If something here fails or is slow, that says nothing about Shears on a Mac.

**The CUDA build** is prebuilt in CI (on a machine with no GPU) and downloaded here. If it will not start the notebook prints why, with the diagnostics, and uses the CPU build; please open an issue with that output. With the CPU build the `gemma` mode stops before the 16 GB download, because the real model would take far too long (set `ALLOW_GEMMA_ON_CPU = True` to go on anyway). The first block of the real run also compiles GPU kernels at run time (several minutes); MLX keeps them, and the last cell saves them so a later session can skip that (`SHEARS_JIT_CACHE_URL`).

**The check against Gemma.** The 200 saved translations come from the full-precision model, while the model Shears healed against is the 4-bit QAT checkpoint, so the two differ slightly even where both are right. The agreement is measured on tokens: Gemma's text is re-tokenized, so a few tokens can differ from what Gemma actually emitted. The healing here is a few steps on 32 sentences, so the pruned model's translations are far from Gemma's; that is the point of the demo, not a result.

**Data and credits.** English sentences and the human Japanese translations: [WMT24++](https://huggingface.co/datasets/google/wmt24pp) (Deutsch et al., 2025, [arXiv:2502.12404](https://arxiv.org/abs/2502.12404)), Apache 2.0, config `en-ja_JP`, with the canary and bad-source rows dropped. Gemma's translations: Gemma 4 26B-A4B (Apache 2.0) through OpenRouter, provider NextBit, 2026-10-22; see [Notebooks/data/README.md](https://github.com/a1cd/Shears/blob/main/Notebooks/data/README.md).

**Time, when measured.** `tiny`: on a Mac (M3 Max) a cold setup takes about 100 seconds once Xcode is ready, and the pipeline 2 to 15 seconds; on Linux the setup is a download of a few tens of megabytes. `gemma` on a Colab T4, in the 2026-10-22 run with shorter training rows and no comparison step: download about 3 minutes, prune and heal about 18, export about 3 and loading plus a short translation about 1.5.

**Where to go next**

* The real thing: a Mac with Apple Silicon, `shears-cli run` on `mlx-community/gemma-4-26B-A4B-it-qat-4bit`, with a `--memory-budget`. See the [command line page](https://github.com/a1cd/Shears/blob/main/docs/cli.md) and the [measured results](https://github.com/a1cd/Shears/blob/main/docs/benchmarks.md).
* Turn the export into a GGUF for llama.cpp: `shears-cli gguf --config config.json`, or serve it: `shears-serve serve --model <export dir> --port 8080`.
* What was verified on Linux, and what was not: [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md). If no prebuilt binary can be set up, the notebook stops in section 2b and says why; that page also describes building `shears-cli` by hand.